# cycle-detection-temp-set — ex3: enumerate ALL cycles reachable from root, not just the first

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cycle-detection-temp-set`. Running the final beacon cell reports progress against the `Backprop: cycle detection via temp set` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Enumerate ALL cycles — multi-SCC view, not just the first

Ex1 returned `True` on any back-edge; ex2 returned ONE cycle path.
The deepening move is to find EVERY distinct cycle reachable from
`root`. A graph with two disjoint cycles `(a→b→a)` and `(c→d→e→c)`
should produce a list of TWO paths.

```python
def all_cycles(root, get_children):
    cycles = []           # accumulator across the whole walk
    on_stack = set()
    perm = set()
    path = []
    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in on_stack:
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            cycles.append(path[i:] + [node])
            return         # do NOT halt — keep exploring other branches
        on_stack.add(nid); path.append(node)
        for child in get_children(node):
            visit(child)
        on_stack.discard(nid); path.pop()
        perm.add(nid)
    visit(root)
    return cycles
```

**Critical change from ex2: don't return on back-edge.** Ex2 returned
the first cycle and aborted. To enumerate ALL cycles, the back-edge
case must RECORD and CONTINUE — explore siblings of the current node.
Forgetting this is the universal bug.

**Dedup is the user's problem.** Two paths that traverse the same
cycle starting from different nodes are technically distinct here.
The drill returns paths as-discovered; canonicalization (rotate to
smallest-id-first) is left to the caller — out of scope for the
core enumeration loop.

### Exercise 3 — enumerate ALL cycles reachable from root, not just the first

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the temp-set DFS cycle-detection pattern to enumerate EVERY cycle reachable from `root` (recording each back-edge path and CONTINUING the walk), rather than returning on the first one found.
> Keywords: cycle-enumeration, all-cycles, back-edge, dfs, diagnostic
> ```

**KCs targeted:** `cycle-detection-temp-set`, `continue-after-back-edge`

Implement `ex3_all_cycles(root, get_children)`. Same input contract as ex1/ex2: a `root` node and a `get_children` callable returning the node's direct successors.

Return: `list[list[node]]`. Each inner list is one cycle, with the back-edge target REPEATED at the end (so `[a, b, c, a]` closes the cycle). Empty list `[]` if no cycles exist.

**Critical difference from ex2.** Ex2 returned the first cycle and aborted the walk. Ex3 must RECORD the cycle and CONTINUE — explore the rest of the graph so disjoint cycles are also found.

Algorithm:

```python
def ex3_all_cycles(root, get_children):
    cycles = []
    on_stack = set()
    perm = set()
    path = []
    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in on_stack:
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            cycles.append(path[i:] + [node])
            return  # do NOT halt the walk
        on_stack.add(nid); path.append(node)
        for child in get_children(node):
            visit(child)
        on_stack.discard(nid); path.pop()
        perm.add(nid)
    visit(root)
    return cycles
```

Use `id(node)` for set membership. The test nodes are dataclass-like and don't override `__hash__`/`__eq__` in a way that conflicts.

Do NOT canonicalize / dedup paths — return them in the order discovered.

In [ ]:
def ex3_all_cycles(root, get_children) -> list:
    """Return list of all cycle paths reachable from root; [] if none."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # Node with `children: list` and `name: str` for readable diffs.
        class N:
            def __init__(self, name):
                self.name = name
                self.children = []
            def __repr__(self):
                return f'N({self.name})'

        def kids(n):
            return n.children

        # === Pure DAG — no cycles ===
        a, b, c, d = N('a'), N('b'), N('c'), N('d')
        a.children = [b, c]
        b.children = [d]
        c.children = [d]
        assert ex3_all_cycles(a, kids) == [], 'diamond DAG has no cycles'

        # === Single cycle ===
        a, b, c = N('a'), N('b'), N('c')
        a.children = [b]
        b.children = [c]
        c.children = [a]
        cycles = ex3_all_cycles(a, kids)
        assert len(cycles) == 1, f'expected 1 cycle; got {len(cycles)}: {cycles}'
        names = [n.name for n in cycles[0]]
        assert names == ['a', 'b', 'c', 'a'], f'cycle path wrong: {names}'

        # === Self-loop ===
        a = N('a')
        a.children = [a]
        cycles = ex3_all_cycles(a, kids)
        assert len(cycles) == 1
        assert [n.name for n in cycles[0]] == ['a', 'a']

        # === Two disjoint cycles (the headline test) ===
        # Root r has two children r->a and r->c. a forms cycle a->b->a, c forms cycle c->d->e->c.
        r, a, b, c, d, e = N('r'), N('a'), N('b'), N('c'), N('d'), N('e')
        r.children = [a, c]
        a.children = [b]
        b.children = [a]      # cycle 1: a-b-a
        c.children = [d]
        d.children = [e]
        e.children = [c]      # cycle 2: c-d-e-c
        cycles = ex3_all_cycles(r, kids)
        assert len(cycles) == 2, f'expected 2 disjoint cycles; got {len(cycles)}: {cycles}'
        sigs = sorted([''.join(n.name for n in cyc) for cyc in cycles])
        assert sigs == ['aba', 'cdec'], f'cycle signatures wrong: {sigs}'

        # === Nested: cycle inside a cycle (two back-edges) ===
        # a -> b -> c -> b (inner cycle), c -> a (outer cycle)
        a, b, c = N('a'), N('b'), N('c')
        a.children = [b]
        b.children = [c]
        c.children = [b, a]   # two back-edges from c
        cycles = ex3_all_cycles(a, kids)
        assert len(cycles) == 2, f'expected 2 cycles; got {len(cycles)}: {cycles}'
        sigs = sorted([''.join(n.name for n in cyc) for cyc in cycles])
        # c.children = [b, a]: visit() at c first recurses into b (back-edge → 'bcb'),
        # then recurses into a (back-edge → 'abca'). Both cycles found.
        assert sigs == ['abca', 'bcb'], f'expected cycles abca + bcb; got {sigs}'

        # === The CONTINUE-not-RETURN check: after finding cycle 1, the walk must explore further. ===
        # If implementer returns on first back-edge, only 1 cycle will be found here.
        r, x, y, z = N('r'), N('x'), N('y'), N('z')
        r.children = [x, z]
        x.children = [y]
        y.children = [x]    # cycle x-y-x (found first)
        z.children = [z]    # cycle z-z (found ONLY if walk continues)
        cycles = ex3_all_cycles(r, kids)
        assert len(cycles) == 2, (
            f'must continue walking after finding first cycle; got {len(cycles)}'
        )
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_all_cycles(root, get_children):
    cycles = []
    on_stack = set()
    perm = set()
    path = []

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in on_stack:
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            cycles.append(path[i:] + [node])
            return  # CONTINUE the outer walk — do not halt
        on_stack.add(nid)
        path.append(node)
        for child in get_children(node):
            visit(child)
        on_stack.discard(nid)
        path.pop()
        perm.add(nid)

    visit(root)
    return cycles
```

**The one-line bug ex2-vs-ex3.** Ex2's `find_cycle` returned the found path immediately. Ex3 must APPEND and KEEP WALKING — the back-edge case records the cycle but lets siblings get explored. Forgetting this is the universal mistake when going from find-one to find-all.

**`on_stack.discard` + `path.pop` MUST both run on the way up.** Sibling-subtree contamination otherwise: leftover `on_stack` entries cause false positives on subtrees that share ancestors.

**Don't dedup — leave that to the caller.** Two paths that traverse the same cycle starting from different nodes are technically distinct under this algorithm. Canonicalization (rotate to smallest-id-first, etc.) is a separate concern from discovery.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()